# LogiOn AI 노트북: 데이터 생성 → 의도분류 학습 → function calling 학습 → test 평가 → 파이프라인 체험

편의점 납품기사의 말 한 줄을 받아서 **무엇을 할지(fn)** 와 **필요한 값(점포·품목·문제·수량·인수자)** 을 뽑고, 실행 결과를 문장으로 돌려주는 흐름을 한 노트북에서 끝까지 돌립니다.

| 단계 | 내용 | 결과물 |
| --- | --- | --- |
| 1 | 합성 데이터 생성 (템플릿 + 제미나이 생성분 병합) | `data/intent_dataset.csv`, `data/fc_dataset.csv` |
| 2 | 의도분류 모델 학습 (`klue/roberta-small`) | `out/intent_model/` |
| 3 | function calling 학습 (소형 LLM + LoRA) | `out/fc_lora/`, `out/fc_merged/` (RKLLM 변환 입력) |
| 4 | test 세트로 두 모듈 성능 확인 | 정확도, F1, JSON 유효율, 값 정확도 |
| 5 | 완성 파이프라인 + 직접 입력 테스트 | 문장 입력 → 응답 |
| 6 | (선택) 음성 파일 → STT → 파이프라인 | SenseVoice (sherpa-onnx) |

**실행 환경**: 1단계와 5단계의 규칙 부분은 CPU에서도 돕니다. 2~4단계는 GPU가 필요합니다 (강의 기준 A100 서버, Colab T4도 가능).

**두 CSV의 관계**: 원본은 하나(발화 + 정답 JSON)이고, 의도분류용은 정답 JSON의 `fn`만, function calling용은 JSON 전체를 씁니다. train/val/test 분할도 두 파일이 똑같습니다.

## 0. 설치와 설정

In [1]:
# GPU 서버에서 처음 한 번만 실행 (CPU에서 데이터만 만들 때는 건너뛰어도 됩니다)
# !pip install -q -U "transformers>=4.46" peft accelerate datasets scikit-learn pandas

In [2]:
import os, re, json, random, time, difflib, gc
from collections import Counter
import pandas as pd

SEED = 42
random.seed(SEED)
ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."  # notebooks/에서 열어도 레포 루트 기준으로 읽고 씀
DATA_DIR, OUT_DIR = f"{ROOT}/data", f"{ROOT}/out"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

FUNCS = ["next_stop", "stop_items", "complete", "report_issue", "scan", "other"]

# 학습, 평가, 캘리브레이션, 보드 추론에서 이 문장을 그대로 사용합니다 (바꾸면 다시 학습해야 함)
SYSTEM_PROMPT = (
    "편의점 납품기사의 말을 함수 호출 JSON 한 줄로 바꿔라. 말한 값만 그대로 쓰고 말하지 않은 값은 쓰지 마라.\n"
    "함수: next_stop(), stop_items(store?,item?), complete(store?,receiver?), "
    "report_issue(store?,item?,issue?,qty?,unit?), scan(), other()\n"
    "issue는 파손|수량부족|오배송|온도이상 중 하나, qty는 정수."
)

# 데이터 양 설정
N_PER_FUNC = {"train": 300, "val": 60, "test": 60}       # 템플릿 데이터, 함수별 건수
MIX = {"clear": 0.65, "ask": 0.20, "hard": 0.15}          # 명확 / 되물어야 함 / 헷갈리는 표현
GEMINI_FILE = f"{DATA_DIR}/gemini_raw.jsonl"              # 제미나이가 만든 데이터 (없으면 건너뜀)
GEMINI_SPLIT = {"train": 0.6, "val": 0.1, "test": 0.3}    # 제미나이 데이터는 test에 넉넉히 넣어 일반화 확인
CURRENT_STORE = "역삼점"                                    # 앱이 알려주는 '지금 있는 점포' (점포를 말하지 않으면 이 값)

## 1. 데이터 생성

### 1-1. 가상 DB와 말 재료
점포, 품목, 인수자는 가상 DB 값입니다. 품목은 실제로 부르는 이름이 여러 개라서 `ITEM_FORMS`에 "DB 이름: 말하는 형태들"로 적었습니다. "우유", "도시락", "삼각김밥"처럼 여러 품목에 걸리는 말은 자연스럽게 **되묻기(ask)** 데이터가 됩니다.

In [3]:
STORES = ["역삼점", "서초점", "방배점", "반포점", "도곡점", "대치점", "수서점", "논현점",
          "신사점", "청담점", "삼성점", "석촌점", "가락점", "문정점", "송파점", "잠실점",
          "개포점", "일원점", "압구정점", "잠원점", "가산점", "구로점", "금천점", "신림점"]

ITEM_FORMS = {
    "서울우유": ["서울우유", "서울우유", "우유"],
    "바나나우유": ["바나나우유", "바나나우유", "바나나 우유"],
    "딸기우유": ["딸기우유", "딸기 우유"],
    "요거트": ["요거트", "요구르트"],
    "제육도시락": ["제육도시락", "제육 도시락", "도시락"],
    "치킨마요도시락": ["치킨마요도시락", "치킨마요", "마요 도시락"],
    "참치삼각김밥": ["참치삼각김밥", "참치김밥", "삼각김밥"],
    "스팸마요삼각김밥": ["스팸마요삼각김밥", "스팸마요", "삼각김밥"],
    "샌드위치": ["샌드위치"],
    "생수": ["생수", "물"],
    "콜라": ["콜라"],
    "사이다": ["사이다"],
    "이온음료": ["이온음료", "이온 음료"],
    "캔커피": ["캔커피", "커피"],
    "새우깡": ["새우깡"],
    "감자칩": ["감자칩"],
    "초코파이": ["초코파이"],
    "젤리": ["젤리"],
    "컵라면": ["컵라면", "라면"],
    "아이스크림": ["아이스크림"],
}
ITEMS = list(ITEM_FORMS)
RECEIVERS = ["점장님", "직원분", "매니저님", "알바생", "사장님"]

ISSUES = ["파손", "수량부족", "오배송", "온도이상"]
ISSUE_PHRASES = {
    "파손": ["터졌어", "깨졌어", "찌그러졌어", "박스 눌렸어", "터짐", "파손됐어", "새고 있어", "다 찢어졌어", "박살났어"],
    "수량부족": ["모자라", "수량이 안 맞아", "비어있어", "부족해", "모자람", "덜 왔어", "개수가 안 맞아", "하나 빠졌어"],
    "오배송": ["다른 게 왔어", "잘못 왔어", "주문이랑 달라", "오배송이야", "엉뚱한 게 실렸어", "이거 우리 거 아니래"],
    "온도이상": ["다 녹았어", "미지근해", "상했어", "녹았음", "온도가 높아", "차갑지가 않아", "냉장이 안 됐어"],
}
UNITS = ["개", "개", "개", "박스", "판", "봉지"]
KOR_NUM = {1: "한", 2: "두", 3: "세", 4: "네", 5: "다섯", 6: "여섯", 7: "일곱", 8: "여덟", 9: "아홉", 10: "열"}

### 1-2. 말 틀(템플릿)
함수마다 일반 틀 약 26개와 헷갈리는 틀 6개를 둡니다. 맨 앞에 `!`가 붙은 틀은 **hard**(다른 기능이나 잡담과 헷갈리기 쉬운 표현)입니다.

틀은 함수별로 train/val/test에 **나눠서 배정**하므로, val과 test 문장은 학습 때 본 적 없는 틀에서 나옵니다. 그래서 "템플릿이 조금 달라지면 틀리지 않을까"를 test 점수로 직접 확인할 수 있습니다.

In [4]:
TEMPLATES = {
    "next_stop": [
        "다음 어디야", "다음 점포 알려줘", "다음은 어디로 가", "다음 목적지 뭐야", "몇 군데 남았어",
        "남은 배송지 몇 개야", "오늘 진행상황 어때", "지금까지 몇 군데 했어", "다음 가야 할 곳 알려줘",
        "다음 배송지 말해줘", "이제 어디로 가면 돼", "오늘 몇 곳 더 가야 돼", "다음 납품처 어디지",
        "남은 곳 몇 개 더 있어", "다음 코스 알려줘", "이제 어디야", "다음 어디 가야 하지", "남은 점포 알려줘",
        "오늘 얼마나 했지", "다음 행선지 알려줘", "다음 들를 데 어디야", "남은 코스 몇 개야",
        "다음 점포 어디였지", "어디 가야 되더라", "오늘 남은 일정 알려줘", "몇 개 남았지",
        "!여기 끝나면 어디 가", "!그 다음은", "!다음 거", "!이제 뭐 하면 돼",
        "!다음 꺼 어디 있는 데야", "!어디로 갈지 알려줘",
    ],
    "stop_items": [
        "여기 뭐 내려야 돼", "{store} 납품 목록 뭐야", "이번 점포 품목 알려줘", "여기 {item} 몇 개야",
        "{item} 몇 개 내리면 돼", "{store}에 뭐 가져가야 하지", "오늘 여기 물건 뭐야", "{item} 수량 얼마야",
        "여기 내릴 거 읽어줘", "{store} {item} 몇 박스야", "이 점포 납품 품목 뭐였지", "내릴 물건 알려줘",
        "{store} 물건 뭐 내려", "{item} 얼마나 내려야 해", "여기 {item} 몇 개 가져가", "{store} 오늘 뭐 내려",
        "{store} 내릴 거 알려줘", "{item} 몇 개였지", "여기 납품 물량 얼마야", "{store} 거 뭐 있었지",
        "{store} 리스트 불러줘", "여기 들어갈 거 뭐뭐야", "{store} 물량 확인해줘", "{item} 여기 몇 개 들어가",
        "이 집 뭐 내리더라", "{store} 납품할 거 몇 가지야",
        "!{item} 여기 들어가는 거 맞아", "!{store} 거 다 실렸나", "!{item} 있어", "!여기 뭐였더라",
        "!{store} 꺼 좀 봐줘", "!{item} 내리는 거야",
    ],
    "complete": [
        "여기 납품 끝났어", "{store} 배송 완료", "다 내렸어", "이번 점포 끝", "{receiver}한테 넘겼어",
        "인수인계 끝났어", "{store} 납품 완료 처리해줘", "{receiver} 받았어", "여기 마무리했어", "하차 끝났어",
        "전달 완료했어", "여기 다 됐어", "{store} 끝", "{receiver}한테 전달했어", "{store} 다 내렸어 {receiver} 받았어",
        "납품 완료야", "{store} 하차 완료", "여기 인수인계 마쳤어", "{receiver}한테 인계했어",
        "{store} 납품 다 했어", "{receiver} 사인 받았어", "{receiver} 확인했어 완료", "완료 찍어줘",
        "여기 완료 처리", "배송 완료 눌러줘", "{store} 마감",
        "!끝났어 다음", "!{receiver}가 확인했대", "!여기 정리됐어", "!{store} 오케이",
        "!다 넣어놨어", "!{receiver}한테 줬어",
    ],
    "report_issue": [
        "{item} {qty} {issue}", "{store} {item} {issue}", "{item} {issue}", "여기 {item} {qty} {issue}",
        "{item} {qty}쯤 {issue}", "{store}에 {item} 보냈는데 {issue}", "물건 {issue} 기록해줘",
        "{item} 문제 있어 {issue}", "이상 있어 {item} {issue}", "{item} {qty} 이상 생겼어 {issue}",
        "불량 기록 {item} {issue}", "{item} 상태 안 좋아 {issue}", "{qty} {issue}", "{store} {item} {qty} {issue}",
        "{item} 좀 {issue}", "{item} 문제 있어", "{store}에서 {item} {issue}", "기록해줘 {item} {qty} {issue}",
        "{item} {issue} {qty}", "아 {item} {issue} 어떡해", "{item} {issue} 기록 좀", "{qty} 정도 {issue} {item}",
        "이거 {issue}", "{item} 쪽에 문제가 좀 있어", "{item} 박스 하나 {issue}", "{store} {item} {qty} {issue} 등록해줘",
        "!{item} 이상한데", "!{item} 이거 맞나", "!이거 좀 문제인데", "!{item} {issue} 같기도 하고",
        "!{item} 상태 좀 봐야겠다", "!{item} 좀 애매한데 {issue}",
    ],
    "scan": [
        "스캔 켜줘", "바코드 찍을게", "검수 모드 켜줘", "카메라 켜줘", "바코드 스캔 시작", "스캔할게",
        "물건 확인할게", "검수 시작", "스캐너 켜", "바코드 인식 켜줘", "상품 찍어볼게", "스캐너 열어줘",
        "바코드 읽을게", "카메라로 확인할래", "물품 스캔 켜줘", "스캔 모드로 바꿔줘", "검수 할게", "바코드 켜",
        "스캔 좀 해볼게", "송장 찍을게", "QR 찍을게", "검품 시작할게", "바코드 좀 확인할게", "카메라 좀 열어",
        "이거 찍어볼게", "라벨 찍을게",
        "!이거 뭔지 확인해봐", "!찍어", "!이거 맞는지 봐줘", "!카메라", "!확인 좀 해볼게", "!이거 읽어봐",
    ],
    "other": [
        "오늘 날씨 어때", "노래 틀어줘", "지금 몇 시야", "점심 뭐 먹지", "주유소 어디 있어", "라디오 켜줘",
        "뉴스 틀어줘", "졸리다", "음악 틀어줘", "커피 마시고 싶다", "오늘 몇 월 며칠이야", "차 막히네",
        "에어컨 켜줘", "농담 하나 해줘", "근처 화장실 어디야", "배고프다", "내일 비 와", "음 저기", "어 잠깐만",
        "고마워", "수고했어", "오늘 힘들다", "볼륨 좀 키워줘", "퇴근하고 싶다", "엄마한테 전화해줘", "아니야 됐어",
        "!{item} 맛있겠다", "!{store} 점장님 친절하시더라", "!어제 {store} 가는 길 막혔어",
        "!{item} 요즘 잘 팔리나", "!오늘 납품 많네", "!{receiver} 오늘 안 계시네",
    ],
}
for f in FUNCS:
    assert len(TEMPLATES[f]) >= 30, f

FILLERS_PRE = ["", "", "", "", "아 ", "저기 ", "음 ", "어 ", "그 ", "자 ", "이제 ", "잠깐 "]
FILLERS_POST = ["", "", "", "", "", " 좀", " 지금", " 빨리", " 얼른"]
SIMILAR = {"역": "약", "점": "졈", "삼": "산", "우": "오", "개": "게", "박": "밖", "터": "더", "서": "소"}

### 1-3. 값 보정 규칙 (모델 아님)
LLM은 말한 값을 그대로 옮기기만 하고, DB 이름 맞추기와 되묻기는 이 규칙 코드가 합니다. 데이터 생성에서는 이 규칙으로 각 문장이 **명확(clear)** 한지 **되물어야(ask)** 하는지 판정하고, 5단계 파이프라인에서도 같은 코드를 씁니다.

In [5]:
NEEDS = {"next_stop": [], "scan": [], "other": [], "stop_items": ["store"],
         "complete": ["store"], "report_issue": ["store", "item", "issue"]}


def parse_call(raw: str) -> dict:
    """모델 출력에서 첫 JSON 객체를 꺼낸다. 실패하면 {"fn":"other","_bad":True}."""
    m = re.search(r"\{.*?\}", str(raw).replace("\n", " "))
    if not m:
        return {"fn": "other", "_bad": True}
    try:
        d = json.loads(m.group(0))
    except json.JSONDecodeError:
        return {"fn": "other", "_bad": True}
    if not isinstance(d, dict) or d.get("fn") not in NEEDS:
        return {"fn": "other", "_bad": True}
    return d


def _jamo(s):
    out = []
    for ch in s:
        c = ord(ch) - 0xAC00
        out += [c // 588, 100 + (c % 588) // 28, 200 + c % 28] if 0 <= c < 11172 else [1000 + ord(ch)]
    return out


def similarity(a, b):
    return difflib.SequenceMatcher(None, _jamo(a), _jamo(b)).ratio()


def build_aliases(item_forms):
    d = {}
    for full, forms in item_forms.items():
        for f in forms:
            d.setdefault(f.replace(" ", ""), set()).add(full)
    return {k: sorted(v) for k, v in d.items()}


ALIASES = build_aliases(ITEM_FORMS)


def snap(value, names, aliases=None, cutoff=0.72):
    """value에 맞는 DB 이름 후보. 1개면 확정, 여러 개면 되묻기, 0개면 못 찾음."""
    v = str(value).replace(" ", "")
    if not v:
        return []
    plain = {n.replace(" ", ""): n for n in names}
    if v in plain:
        return [plain[v]]
    got = [n for n in (aliases or {}).get(v, []) if n in names]
    if len(v) >= 2:
        got += [n for k, n in plain.items() if v in k and n not in got]
    if got:
        return got
    scored = sorted(((similarity(v, k), n) for k, n in plain.items()), reverse=True)
    if scored and scored[0][0] >= cutoff:
        top = scored[0][0]
        return [n for s, n in scored if s >= top - 0.05][:3]
    return []


def _pick(label, value, names, aliases=None):
    c = snap(value, names, aliases)
    if len(c) == 1:
        return c[0], None
    if not c:
        return None, f"{label} 못 찾았어요. 다시 말씀해 주세요."
    return None, f"{', '.join(c[:3])} 중 어떤 건가요?"


def resolve(call, current_store=None):
    """{"ok":True, fn, store?, item?, ...} 또는 {"ok":False, "ask": 되묻는 문장, "slot": 되묻는 값}"""
    fn = call.get("fn", "other")
    out = {"ok": True, "fn": fn}
    if fn in ("next_stop", "scan", "other"):
        return out
    if call.get("store"):
        store, ask = _pick("점포를", call["store"], STORES)
        if ask:
            return {"ok": False, "fn": fn, "ask": ask, "slot": "store"}
    else:
        store = current_store
    if "store" in NEEDS[fn] and not store:
        return {"ok": False, "fn": fn, "ask": "어느 점포인지 말씀해 주세요.", "slot": "store"}
    out["store"] = store
    if call.get("item"):
        item, ask = _pick("품목을", call["item"], ITEMS, ALIASES)
        if ask:
            return {"ok": False, "fn": fn, "ask": ask, "slot": "item"}
        out["item"] = item
    elif "item" in NEEDS[fn]:
        return {"ok": False, "fn": fn, "ask": "어떤 품목인가요?", "slot": "item"}
    if fn == "report_issue":
        if call.get("issue") not in ISSUES:
            return {"ok": False, "fn": fn, "ask": "파손, 수량부족, 오배송, 온도이상 중 어떤 문제인가요?", "slot": "issue"}
        out["issue"] = call["issue"]
        if isinstance(call.get("qty"), int):
            out["qty"], out["unit"] = call["qty"], call.get("unit", "개")
    if fn == "complete" and call.get("receiver"):
        out["receiver"] = call["receiver"]
    return out


print(resolve({"fn": "report_issue", "item": "우유", "issue": "파손", "qty": 2, "unit": "개"}, CURRENT_STORE))
print(resolve({"fn": "complete", "store": "약삼졈"}))

{'ok': False, 'fn': 'report_issue', 'ask': '서울우유, 바나나우유, 딸기우유 중 어떤 건가요?', 'slot': 'item'}
{'ok': True, 'fn': 'complete', 'store': '역삼점'}


### 1-4. 템플릿 데이터 만들기
틀에 값을 채우면 정답 JSON이 자동으로 만들어집니다. 말투(반말/존댓말), 군더더기 말, STT 오인식 흉내(약 10%)를 섞습니다. 그 뒤 함수·분할별로 clear / ask / hard 비율(`MIX`)에 맞춰 뽑습니다. 되묻기가 생길 수 없는 함수(next_stop, scan, other)는 ask 몫을 clear로 돌립니다.

In [6]:
def to_polite(t):
    for a, b in [("줘", "주세요"), ("야", "예요"), ("돼", "돼요"), ("어", "어요"), ("해", "해요"), ("게", "게요"), ("지", "죠")]:
        if t.endswith(a):
            return t[: -len(a)] + b
    return t


def stt_noise(t, rng, rate=0.08):
    return "".join(SIMILAR[c] if c in SIMILAR and rng.random() < rate else c for c in t)


def render(func, tmpl, rng):
    tmpl = tmpl.lstrip("!")
    vals, args = {}, {}
    if "{store}" in tmpl:
        vals["store"] = args["store"] = rng.choice(STORES)
    if "{item}" in tmpl:
        vals["item"] = args["item"] = rng.choice(rng.choice(list(ITEM_FORMS.values())))
    if "{issue}" in tmpl:
        enum = rng.choice(ISSUES)
        vals["issue"], args["issue"] = rng.choice(ISSUE_PHRASES[enum]), enum
    if "{qty}" in tmpl:
        n = rng.choice([1, 2, 2, 3, 3, 4, 5, 6, 8, 10, 12])
        unit = rng.choice(UNITS)
        r = rng.random()
        vals["qty"] = f"{n}{unit}" if r < 0.45 or n not in KOR_NUM else (f"{KOR_NUM[n]} {unit}" if r < 0.8 else f"{KOR_NUM[n]}{unit}")
        args["qty"], args["unit"] = n, unit
    if "{receiver}" in tmpl:
        vals["receiver"] = args["receiver"] = rng.choice(RECEIVERS)
    text = tmpl.format(**vals)
    if rng.random() < 0.35:
        text = to_polite(text)              # 존댓말이면 뒤에 군더더기를 붙이지 않음
    else:
        text += rng.choice(FILLERS_POST)
    if rng.random() < 0.10:
        text = stt_noise(text, rng)       # 말은 틀려도 정답 값은 그대로 (오인식 -> 올바른 값)
    text = (rng.choice(FILLERS_PRE) + text).strip()
    call = {"fn": func}
    if func != "other":                   # other는 값을 쓰지 않음
        for k in ["store", "item", "issue", "qty", "unit", "receiver"]:
            if k in args:
                call[k] = args[k]
    return text, call


def ambiguity_of(call, hard):
    if hard:
        return "hard"
    return "clear" if resolve(call, CURRENT_STORE)["ok"] else "ask"


def split_templates(tms, rng):
    """일반 틀과 hard 틀을 각각 train/val/test로 나눔 (같은 틀이 두 분할에 들어가지 않게)."""
    out = {"train": [], "val": [], "test": []}
    for group in ([t for t in tms if not t.startswith("!")], [t for t in tms if t.startswith("!")]):
        g = group[:]
        rng.shuffle(g)
        k = max(1, round(len(g) * 0.15))
        out["test"] += g[:k]
        out["val"] += g[k:2 * k]
        out["train"] += g[2 * k:]
    return out


def make_template_data(rng):
    rows, used = [], set()
    tsplit = {}
    for func in FUNCS:
        tsplit[func] = split_templates(TEMPLATES[func], rng)
        for split, n in N_PER_FUNC.items():
            pool = {"clear": [], "ask": [], "hard": []}
            for _ in range(n * 40):
                tmpl = rng.choice(tsplit[func][split])
                text, call = render(func, tmpl, rng)
                if text in used:
                    continue
                used.add(text)
                pool[ambiguity_of(call, tmpl.startswith("!"))].append((text, call))
                if all(len(v) >= n for v in pool.values()):
                    break
            want = {k: int(n * r) for k, r in MIX.items()}
            take = {k: min(want[k], len(pool[k])) for k in pool}
            take["clear"] = min(len(pool["clear"]), n - take["ask"] - take["hard"])  # 모자란 몫은 clear로
            for amb, k in take.items():
                for text, call in pool[amb][:k]:
                    rows.append({"text": text, "call": call, "split": split, "source": "template", "ambiguity": amb})
    return rows, tsplit


rng = random.Random(SEED)
template_rows, TEMPLATE_SPLIT = make_template_data(rng)
print(len(template_rows), "rows")
print(pd.DataFrame(template_rows).groupby(["split", "ambiguity"]).size().unstack(fill_value=0))

2520 rows
ambiguity  ask  clear  hard
split                      
test        12    294    54
train      120   1410   270
val         24    282    54


### 1-5. 제미나이 생성 데이터 합치기 (선택)
템플릿만 쓰면 문장 모양이 한정되므로, 전체의 20~30%는 제미나이에게 자유롭게 만들게 해서 섞는 것을 권장합니다. 아래 셀이 프롬프트를 `data/gemini_prompt.txt`로 저장합니다. 이 프롬프트를 제미나이에 붙여넣고, 받은 결과를 `data/gemini_raw.jsonl`로 저장한 뒤 다시 실행하면 자동으로 검사해서 합칩니다.

검사 규칙: JSON 형식, 함수 이름, issue 값, qty 정수, 그리고 **정답에 적힌 점포·품목·인수자가 실제 문장에 들어 있는지** 확인합니다. 하나라도 어긋나면 버립니다. 제미나이 데이터는 test에 30%를 넣어서 "우리 템플릿이 아닌 문장"에 대한 성능을 따로 볼 수 있게 했습니다.

In [7]:
GEMINI_PROMPT = f"""너는 한국어 음성 비서 학습 데이터를 만드는 사람이다.
상황: 편의점에 물건을 납품하는 트럭 기사가 운전하거나 짐을 내리면서 음성 비서에게 말한다.
기사의 말 한 문장과, 그 말을 함수 호출로 바꾼 정답 JSON을 만들어라.

[함수]
- next_stop: 다음 배송지나 남은 배송지를 물을 때. 값 없음.
- stop_items: 점포에 내릴 품목이나 수량을 물을 때. store, item (말한 경우만)
- complete: 납품을 마쳤다고 알릴 때. store, receiver (말한 경우만)
- report_issue: 물건에 문제가 있다고 알릴 때. store, item, issue, qty, unit (말한 경우만)
  issue는 파손 | 수량부족 | 오배송 | 온도이상 중 하나 (말 속의 표현을 이 넷 중 하나로 바꿔 적는다)
  qty는 정수 (두 개 -> 2), unit은 개 | 박스 | 판 | 봉지 중 하나
- scan: 바코드/송장/카메라 스캔을 켜달라고 할 때. 값 없음.
- other: 위 다섯 가지가 아닌 모든 말 (잡담, 날씨, 음악, 혼잣말 등). 값 없음.

[값 규칙]
- 문장에 실제로 나온 말만 그대로 적는다. 말하지 않은 값은 키를 아예 쓰지 않는다.
- store는 다음 중 하나만 사용: {", ".join(STORES)}
- item은 다음 표현 중 하나를 문장에 쓰고, 문장에 쓴 표현 그대로 적는다: {", ".join(sorted({f for v in ITEM_FORMS.values() for f in v}))}
- receiver는 다음 중 하나: {", ".join(RECEIVERS)}

[다양성 요구]
- 실제 기사가 말하듯 자연스럽게: 반말, 존댓말, 짧은 말, 혼잣말 섞인 말, 말 끊김, 사투리 약간, 군더더기("아", "그", "저기")
- 같은 뜻이라도 단어와 어순을 매번 다르게. 비슷한 문장 반복 금지.
- 일부러 애매한 말도 20% 섞어라 (예: 품목을 "우유"처럼 여러 개에 걸리게 말함, 품목이나 문제 종류를 빼먹음).
- other에는 납품 단어가 섞였지만 명령이 아닌 말도 넣어라 (예: "역삼점 점장님 오늘 기분 좋으시네").

[출력 형식]
한 줄에 하나씩 JSON Lines로만 출력. 설명, 번호, 코드블록 표시는 쓰지 마라.
{{"text": "우유 두 개 터졌어요", "label": {{"fn": "report_issue", "item": "우유", "issue": "파손", "qty": 2, "unit": "개"}}}}
{{"text": "다음 어디 가면 되지", "label": {{"fn": "next_stop"}}}}

함수마다 40개씩, 총 240줄을 만들어라."""

with open(f"{DATA_DIR}/gemini_prompt.txt", "w", encoding="utf-8") as f:
    f.write(GEMINI_PROMPT)
print(GEMINI_PROMPT[:400], "...")

너는 한국어 음성 비서 학습 데이터를 만드는 사람이다.
상황: 편의점에 물건을 납품하는 트럭 기사가 운전하거나 짐을 내리면서 음성 비서에게 말한다.
기사의 말 한 문장과, 그 말을 함수 호출로 바꾼 정답 JSON을 만들어라.

[함수]
- next_stop: 다음 배송지나 남은 배송지를 물을 때. 값 없음.
- stop_items: 점포에 내릴 품목이나 수량을 물을 때. store, item (말한 경우만)
- complete: 납품을 마쳤다고 알릴 때. store, receiver (말한 경우만)
- report_issue: 물건에 문제가 있다고 알릴 때. store, item, issue, qty, unit (말한 경우만)
  issue는 파손 | 수량부족 | 오배송 | 온도이상 중 하나 (말 속의 표현 ...


In [8]:
ALL_ITEM_FORMS = {f.replace(" ", "") for v in ITEM_FORMS.values() for f in v}
ALLOWED_KEYS = {"fn", "store", "item", "issue", "qty", "unit", "receiver"}


def check_gemini(rec):
    """통과하면 (text, call), 아니면 None"""
    text, call = str(rec.get("text", "")).strip(), rec.get("label")
    if not text or not isinstance(call, dict) or call.get("fn") not in FUNCS:
        return None
    if set(call) - ALLOWED_KEYS:
        return None
    if call["fn"] in ("next_stop", "scan", "other") and len(call) > 1:
        return None
    if "issue" in call and call["issue"] not in ISSUES:
        return None
    if "qty" in call and not isinstance(call["qty"], int):
        return None
    flat = text.replace(" ", "")
    for k in ("store", "item", "receiver"):          # 정답 값이 실제 문장에 있어야 함 (지어낸 값 방지)
        if k in call and str(call[k]).replace(" ", "") not in flat:
            return None
    if "store" in call and call["store"] not in STORES:
        return None
    if "item" in call and call["item"].replace(" ", "") not in ALL_ITEM_FORMS:
        return None
    order = ["fn", "store", "item", "issue", "qty", "unit", "receiver"]
    return text, {k: call[k] for k in order if k in call}


def load_gemini(path, rng, seen):
    if not os.path.exists(path):
        print(f"{path} 없음 -> 템플릿 데이터만 사용합니다.")
        return []
    ok, bad = [], 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip().strip(",")
            if not line.startswith("{"):
                continue
            try:
                r = check_gemini(json.loads(line))
            except json.JSONDecodeError:
                r = None
            if r is None or r[0] in seen:
                bad += 1
                continue
            seen.add(r[0])
            ok.append(r)
    rng.shuffle(ok)
    rows, n = [], len(ok)
    cut1, cut2 = int(n * GEMINI_SPLIT["train"]), int(n * (GEMINI_SPLIT["train"] + GEMINI_SPLIT["val"]))
    for i, (text, call) in enumerate(ok):
        split = "train" if i < cut1 else "val" if i < cut2 else "test"
        rows.append({"text": text, "call": call, "split": split, "source": "gemini",
                     "ambiguity": ambiguity_of(call, hard=False)})
    print(f"제미나이 데이터: 통과 {len(ok)}건, 버림 {bad}건")
    return rows


gemini_rows = load_gemini(GEMINI_FILE, random.Random(SEED), {r["text"] for r in template_rows})

제미나이 데이터: 통과 234건, 버림 2건


### 1-6. CSV 2개 저장
- `intent_dataset.csv`: `id, text, label(=fn), split, source, ambiguity` → 의도분류 학습용
- `fc_dataset.csv`: `id, text, target(=정답 JSON 문자열), split, source, ambiguity` → function calling 학습용

In [9]:
master = pd.DataFrame(template_rows + gemini_rows).sample(frac=1, random_state=SEED).reset_index(drop=True)
master["id"] = [f"u{i:05d}" for i in range(len(master))]
master["fn"] = master["call"].map(lambda c: c["fn"])
master["target"] = master["call"].map(lambda c: json.dumps(c, ensure_ascii=False, separators=(",", ":")))

intent_df = master[["id", "text", "fn", "split", "source", "ambiguity"]].rename(columns={"fn": "label"})
fc_df = master[["id", "text", "target", "split", "source", "ambiguity"]]
intent_df.to_csv(f"{DATA_DIR}/intent_dataset.csv", index=False, encoding="utf-8-sig")
fc_df.to_csv(f"{DATA_DIR}/fc_dataset.csv", index=False, encoding="utf-8-sig")

# 누출 검사: 같은 문장이 두 분할에 있으면 안 됨
assert master.groupby("text")["split"].nunique().max() == 1
print("총", len(master), "건")
print(pd.crosstab(master["fn"], master["split"], margins=True))
print(pd.crosstab(master["ambiguity"], master["split"], normalize="columns").round(2))
fc_df.sample(8, random_state=1)

총

 2754 건
split         test  train  val   All
fn                                  
complete        70    325   64   459
next_stop       74    321   63   458
other           70    327   63   460
report_issue    69    325   64   458
scan            73    323   64   460
stop_items      75    319   65   459
All            431   1940  383  2754
split      test  train   val
ambiguity                   
ask        0.04   0.07  0.07
clear      0.84   0.79  0.79
hard       0.13   0.14  0.14


,id,text,target,split,source,ambiguity
1785,u01785,일원점 내릴 거 알려줘,"{""fn"":""stop_items"",""store"":""일원점""}",train,template,clear
2639,u02639,3개 녹았음,"{""fn"":""report_issue"",""issue"":""온도이상"",""qty"":3,""u...",train,template,ask
857,u00857,자 전달 완료했어,"{""fn"":""complete""}",test,template,clear
2530,u02530,반포점 우유 미지근하다 온도이상,"{""fn"":""report_issue"",""store"":""반포점"",""item"":""우유""...",train,gemini,ask
335,u00335,다음 가야 할 곳 알려줘 지금,"{""fn"":""next_stop""}",train,template,clear
367,u00367,이제 음 저기,"{""fn"":""other""}",train,template,clear
1065,u01065,아 일원점 오케이,"{""fn"":""complete"",""store"":""일원점""}",val,template,hard
1755,u01755,그 엄마한테 전화해줘 빨리,"{""fn"":""other""}",test,template,clear


In [10]:
# RKLLM 양자화 캘리브레이션용 발화 (train에서 함수별 균등 450건, 강의 기준 300~1,000건)
calib = (master[master.split == "train"].groupby("fn", group_keys=False)
         .apply(lambda g: g.sample(min(len(g), 75), random_state=SEED)))
with open(f"{DATA_DIR}/calibration_utterances.json", "w", encoding="utf-8") as f:
    json.dump(calib["text"].tolist(), f, ensure_ascii=False, indent=2)
print("calibration", len(calib))

calibration 450


## 2. 의도분류 모델 학습
`intent_dataset.csv`로 문장 → fn(6개 중 하나)을 맞히는 작은 한국어 모델을 학습합니다.

- 모델: `klue/roberta-small` (약 6,800만 파라미터, 한국어 사전학습, 문장 하나에 수 ms)
- 파이프라인에서의 역할: 값이 필요 없는 기능(next_stop, scan, other)은 이 모델만으로 바로 처리하고 LLM을 건너뜁니다. 그래서 자주 쓰는 짧은 명령의 응답이 빨라집니다.
- val 세트의 macro F1이 가장 높은 epoch을 저장합니다.

In [11]:
# ▶ GPU 서버에서 실행
import numpy as np, torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)

set_seed(SEED)
SKIP_TRAIN = False      # True: 학습은 건너뛰고 out/에 저장된 모델을 불러와서 4~5단계만 실행
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

intent_df = pd.read_csv(f"{DATA_DIR}/intent_dataset.csv")
fc_df = pd.read_csv(f"{DATA_DIR}/fc_dataset.csv")
LABEL2ID = {f: i for i, f in enumerate(FUNCS)}
ID2LABEL = {i: f for f, i in LABEL2ID.items()}

INTENT_BASE = "klue/roberta-small"
INTENT_DIR = f"{OUT_DIR}/intent_model"
intent_tok = AutoTokenizer.from_pretrained(INTENT_BASE)


def to_ds(df):
    ds = Dataset.from_pandas(df[["text", "label"]].assign(label=df["label"].map(LABEL2ID)), preserve_index=False)
    return ds.map(lambda b: intent_tok(b["text"], truncation=True, max_length=64), batched=True)


ds_tr, ds_va = (to_ds(intent_df[intent_df.split == s]) for s in ["train", "val"])
print(len(ds_tr), len(ds_va))

device: cuda NVIDIA GeForce RTX 5050


Map:   0%|          | 0/1940 [00:00<?, ? examples/s]

Map:   0%|          | 0/383 [00:00<?, ? examples/s]

1940 383


In [12]:
# ▶ GPU 서버에서 실행
def intent_metrics(p):
    pred = p.predictions.argmax(-1)
    return {"acc": accuracy_score(p.label_ids, pred), "macro_f1": f1_score(p.label_ids, pred, average="macro")}


intent_model = AutoModelForSequenceClassification.from_pretrained(
    INTENT_BASE, num_labels=len(FUNCS), id2label=ID2LABEL, label2id=LABEL2ID)
intent_model.config.torchscript = False  # 체크포인트 config의 torchscript=True가 forward() 출력을 tuple로 바꿔버려서 끔

args = TrainingArguments(
    output_dir=f"{OUT_DIR}/intent_ckpt", num_train_epochs=5, learning_rate=5e-5,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, warmup_ratio=0.1, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="macro_f1", save_total_limit=1, logging_steps=20, seed=SEED, report_to="none",
)
trainer = Trainer(model=intent_model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
                  data_collator=DataCollatorWithPadding(intent_tok), compute_metrics=intent_metrics)
if SKIP_TRAIN and os.path.exists(INTENT_DIR):          # 이미 학습했으면 저장본을 불러옴
    intent_model = AutoModelForSequenceClassification.from_pretrained(INTENT_DIR).to(DEVICE)
    intent_model.config.torchscript = False
    print("저장된 의도분류 모델을 불러왔습니다.")
else:
    t0 = time.time()
    trainer.train()
    print(f"학습 시간 {time.time() - t0:.0f}초", trainer.evaluate())
    trainer.save_model(INTENT_DIR)
    intent_tok.save_pretrained(INTENT_DIR)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-small and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Acc,Macro F1
1,0.260600,0.441151,0.879896,0.873672
2,0.021600,0.646140,0.864230,0.858089
3,0.005900,0.594218,0.874674,0.867144
4,0.004300,0.663017,0.866841,0.859117
5,0.003800,0.657200,0.869452,0.861751


학습 시간 32초 {'eval_loss': 0.4411507248878479, 'eval_acc': 0.8798955613577023, 'eval_macro_f1': 0.8736721721912007, 'eval_runtime': 0.157, 'eval_samples_per_second': 2438.978, 'eval_steps_per_second': 38.209, 'epoch': 5.0}


In [13]:
# ▶ GPU 서버에서 실행
@torch.no_grad()
def intent_predict(texts, model=None, tok=None, bs=64):
    """문장 리스트 -> [(fn, 확률), ...]"""
    model = model or intent_model
    tok = tok or intent_tok
    model.eval()
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], truncation=True, max_length=64, padding=True, return_tensors="pt").to(model.device)
        prob = model(**enc).logits.softmax(-1)
        conf, idx = prob.max(-1)
        out += [(ID2LABEL[int(k)], float(c)) for k, c in zip(idx, conf)]
    return out


print(intent_predict(["우유 두 개 터졌어", "다음 어디야", "노래 틀어줘", "역삼점 뭐 내려"]))

[('report_issue', 0.9426781535148621), ('next_stop', 0.9700732827186584), ('other', 0.9544546008110046), ('stop_items', 0.9339591264724731)]


## 3. function calling 학습 (LoRA)
`fc_dataset.csv`로 문장 → 정답 JSON 한 줄을 출력하도록 소형 LLM을 LoRA로 파인튜닝합니다. **네, 이것도 학습입니다.** 의도분류와 다른 점은 fn뿐 아니라 값까지 글자로 생성한다는 것입니다.

- 기본 모델: `Qwen/Qwen2.5-1.5B-Instruct`. RKLLM 지원 목록(Qwen2/2.5/3/3.5)에 있고, 텍스트 전용 구조라 학습과 변환이 가장 단순합니다.
- 강의 자료의 `Qwen3.5-2B`를 쓰려면 `FC_BASE`만 바꾸면 됩니다. 다만 이 모델은 이미지+텍스트 모델이라 transformers 버전에 따라 불러오는 방식이 다를 수 있으니, 먼저 아래 소량 시험 학습(`SMOKE=True`)으로 확인하세요. `meta-llama/Llama-3.2-1B-Instruct`는 허깅페이스에서 사용 승인을 받아야 합니다.
- loss는 assistant 답(JSON) 부분에만 걸고, system 지시와 사용자 문장 부분은 학습하지 않습니다.
- 강의 순서대로: 소량 시험 학습 → 본 학습 → LoRA 병합 → fp16 저장(RKLLM 변환 입력).

In [14]:
# ▶ GPU 서버에서 실행
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, PeftModel

FC_BASE = "Qwen/Qwen2.5-1.5B-Instruct"     # 또는 "Qwen/Qwen3.5-2B", "meta-llama/Llama-3.2-1B-Instruct"
FC_LORA_DIR, FC_MERGED_DIR = f"{OUT_DIR}/fc_lora", f"{OUT_DIR}/fc_merged"
SMOKE = False                                # True면 64건으로 로딩→학습→저장만 먼저 확인
DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

fc_tok = AutoTokenizer.from_pretrained(FC_BASE)
if fc_tok.pad_token is None:
    fc_tok.pad_token = fc_tok.eos_token


def chat(messages, add_generation_prompt):
    # enable_thinking=False: Qwen3 계열의 생각 모드를 끔 (다른 모델 템플릿은 무시함)
    return fc_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt,
                                      enable_thinking=False)


def build_prompt(text):
    return chat([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}], True)


def build_example(text, target, max_len=160):
    prompt = build_prompt(text)
    full = chat([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text},
                 {"role": "assistant", "content": target}], False)
    if not full.startswith(prompt):            # 템플릿이 다르게 생긴 모델 대비
        full = prompt + target + fc_tok.eos_token
    p_ids = fc_tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = fc_tok(full, add_special_tokens=False)["input_ids"][:max_len]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": labels[:len(f_ids)]}


ex = build_example("우유 두 개 터졌어", '{"fn":"report_issue","item":"우유","issue":"파손","qty":2,"unit":"개"}')
print(fc_tok.decode(ex["input_ids"]))
print("학습되는 부분:", repr(fc_tok.decode([t for t, l in zip(ex["input_ids"], ex["labels"]) if l != -100])))

<|im_start|>system
편의점 납품기사의 말을 함수 호출 JSON 한 줄로 바꿔라. 말한 값만 그대로 쓰고 말하지 않은 값은 쓰지 마라.
함수: next_stop(), stop_items(store?,item?), complete(store?,receiver?), report_issue(store?,item?,issue?,qty?,unit?), scan(), other()
issue는 파손|수량부족|오배송|온도이상 중 하나, qty는 정수.<|im_end|>
<|im_start|>user
우유 두 개 터졌어<|im_end|>
<|im_start|>assistant
{"fn":"report_issue","item":"우유","issue":"파손","qty":2,"unit":"개"}<|im_end|>

학습되는 부분: '{"fn":"report_issue","item":"우유","issue":"파손","qty":2,"unit":"개"}<|im_end|>\n'


In [15]:
# ▶ GPU 서버에서 실행
def fc_rows(split):
    d = fc_df[fc_df.split == split]
    return [build_example(t, y) for t, y in zip(d.text, d.target)]


fc_train, fc_val = fc_rows("train"), fc_rows("val")
if SMOKE:
    fc_train, fc_val = fc_train[:64], fc_val[:16]
lens = [len(r["input_ids"]) for r in fc_train]
print(f"train {len(fc_train)} / val {len(fc_val)} / 토큰 길이 평균 {np.mean(lens):.0f}, 최대 {max(lens)}")
# 최대 길이는 RKLLM 변환 때 max_context(32의 배수)를 정하는 기준이 됩니다.


def collate(batch):
    m = max(len(b["input_ids"]) for b in batch)
    pad = lambda x, v: x + [v] * (m - len(x))
    return {"input_ids": torch.tensor([pad(b["input_ids"], fc_tok.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch])}


def train_fc():
    base = AutoModelForCausalLM.from_pretrained(FC_BASE, torch_dtype=DTYPE, device_map={"": 0})
    lora_cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(base, lora_cfg)
    model.print_trainable_parameters()
    fc_args = TrainingArguments(
        output_dir=f"{OUT_DIR}/fc_ckpt", num_train_epochs=1 if SMOKE else 3, learning_rate=2e-4,
        per_device_train_batch_size=4, per_device_eval_batch_size=8, gradient_accumulation_steps=4,  # 8GB VRAM: 16이면 VRAM 초과로 39초/step, 4면 0.7초/step
        lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=10,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
        bf16=(DTYPE == torch.bfloat16), fp16=(DTYPE == torch.float16), seed=SEED, report_to="none",
        remove_unused_columns=False,
    )
    tr = Trainer(model=model, args=fc_args, train_dataset=fc_train, eval_dataset=fc_val, data_collator=collate)
    t0 = time.time()
    tr.train()
    print(f"학습 시간 {time.time() - t0:.0f}초")
    log_df = pd.DataFrame(tr.state.log_history)
    cols = [c for c in ["epoch", "loss", "eval_loss"] if c in log_df.columns]  # 스텝이 너무 적으면 logging_steps에 걸려 loss가 안 찍힐 수 있음
    subset = [c for c in ["loss", "eval_loss"] if c in cols]
    print(log_df[cols].dropna(how="all", subset=subset).tail(8) if subset else log_df.tail(8))
    model.save_pretrained(FC_LORA_DIR)
    fc_tok.save_pretrained(FC_LORA_DIR)
    del tr  # 옵티마이저 상태를 비워서 다음 셀(merge)이 쓸 VRAM을 확보
    gc.collect()
    torch.cuda.empty_cache()
    return model


FC_LOADED_MERGED = SKIP_TRAIN and os.path.exists(FC_MERGED_DIR)
if FC_LOADED_MERGED:                                     # 이미 학습·병합했으면 저장본을 불러옴
    fc_model = AutoModelForCausalLM.from_pretrained(FC_MERGED_DIR, torch_dtype=DTYPE, device_map={"": 0})
    print("저장된 병합 모델을 불러왔습니다.")
else:
    fc_model = train_fc()

train 1940 / val 383 / 토큰 길이 평균 138, 최대 160


`torch_dtype` is deprecated! Use `dtype` instead!


The model is already on multiple devices. Skipping the move to device specified in `args`.


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


Epoch,Training Loss,Validation Loss
1,0.006300,0.022202
2,0.007800,0.038389
3,0.000300,0.039334


학습 시간 1374초
       epoch    loss  eval_loss
31  2.461856  0.0006        NaN
32  2.544330  0.0003        NaN
33  2.626804  0.0003        NaN
34  2.709278  0.0006        NaN
35  2.791753  0.0002        NaN
36  2.874227  0.0054        NaN
37  2.956701  0.0003        NaN
38  3.000000     NaN   0.039334


In [16]:
# ▶ GPU 서버에서 실행
@torch.no_grad()
def fc_generate(texts, model=None, bs=8, max_new_tokens=64):
    """문장 리스트 -> 모델이 출력한 문자열 리스트 (탐욕적 디코딩, 평가·보드와 같은 조건)"""
    model = model or fc_model
    model.eval()
    fc_tok.padding_side = "left"
    outs = []
    for i in range(0, len(texts), bs):
        prompts = [build_prompt(t) for t in texts[i:i + bs]]
        enc = fc_tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=fc_tok.pad_token_id)
        outs += fc_tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    fc_tok.padding_side = "right"
    return [o.strip() for o in outs]


def fc_predict(text):
    return parse_call(fc_generate([text])[0])


for t in ["우유 두 개 터졌어", "역삼점 뭐 내려야 돼", "점장님한테 넘겼어", "오늘 비 온대"]:
    print(t, "->", fc_generate([t])[0])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


우유 두 개 터졌어 -> {"fn":"report_issue","item":"우유","issue":"파손","qty":2,"unit":"개"}


역삼점 뭐 내려야 돼 -> {"fn":"stop_items","store":"역삼점"}


점장님한테 넘겼어 -> {"fn":"complete","receiver":"점장님"}


오늘 비 온대 -> {"fn":"other"}


In [17]:
# ▶ GPU 서버에서 실행
# LoRA를 원본에 합쳐 fp16 전체 가중치로 저장 -> 이 폴더가 RKLLM 변환(run.sh convert)의 입력
if not FC_LOADED_MERGED:
    merged = fc_model.merge_and_unload()
    merged.to(torch.float16).save_pretrained(FC_MERGED_DIR, safe_serialization=True)
    fc_tok.save_pretrained(FC_MERGED_DIR)
    print("saved:", FC_MERGED_DIR, os.listdir(FC_MERGED_DIR))
    fc_model = merged.to(DTYPE)   # 4단계 평가는 병합된 모델로 (강의: 병합 후 같은 답이 나오는지 확인)

saved: out/fc_merged ['added_tokens.json', 'chat_template.jinja', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']


## 4. test 세트로 두 모듈 성능 확인
test 문장은 학습에 쓰지 않은 **틀**에서 나왔고, 제미나이 데이터가 있으면 그 30%도 들어 있습니다.

- 의도분류: 정확도, macro F1, 함수별 리포트, 혼동표
- function calling: JSON 유효율, fn 정확도, 완전 일치율, 값(slot) F1, **최종 동작 일치율**(보정 규칙까지 거친 결과가 정답과 같은지, 되묻기 여부 포함)
- 두 모듈 모두 `source`(template/gemini)와 `ambiguity`(clear/ask/hard)별로 나눠서 봅니다. 제미나이 쪽 점수가 크게 낮으면 템플릿에 과적합된 것입니다.

In [18]:
# ▶ GPU 서버에서 실행
te_i = intent_df[intent_df.split == "test"].reset_index(drop=True)
t0 = time.time()
pred = intent_predict(te_i.text.tolist())
ms = (time.time() - t0) / len(te_i) * 1000
te_i["pred"], te_i["conf"] = [p for p, _ in pred], [c for _, c in pred]
te_i["correct"] = te_i.pred == te_i.label

print(f"[의도분류] test {len(te_i)}건  정확도 {te_i.correct.mean():.3f}  "
      f"macro F1 {f1_score(te_i.label, te_i.pred, average='macro'):.3f}  ({ms:.1f} ms/문장, GPU 배치 기준)")
print(classification_report(te_i.label, te_i.pred, digits=3))
print(pd.crosstab(te_i.label, te_i.pred, rownames=["정답"], colnames=["예측"]))
print(te_i.groupby(["source", "ambiguity"]).correct.agg(["mean", "count"]).round(3))
te_i[~te_i.correct].head(15)

[의도분류] test 431건  정확도 0.893  macro F1 0.894  (1.9 ms/문장, GPU 배치 기준)
              precision    recall  f1-score   support

    complete      1.000     0.886     0.939        70
   next_stop      0.974     1.000     0.987        74
       other      1.000     0.714     0.833        70
report_issue      0.861     0.986     0.919        69
        scan      0.903     0.890     0.897        73
  stop_items      0.717     0.880     0.790        75

    accuracy                          0.893       431
   macro avg      0.909     0.893     0.894       431
weighted avg      0.908     0.893     0.894       431

예측            complete  next_stop  other  report_issue  scan  stop_items
정답                                                                      
complete            62          0      0             1     7           0
next_stop            0         74      0             0     0           0
other                0          2     50             1     0          17
report_issue         0  

,id,text,label,split,source,ambiguity,pred,conf,correct
6,u00030,근처 화장실 어디야 좀,other,test,template,clear,stop_items,0.520944,False
16,u00056,아 근처 화장실 어디예요,other,test,template,clear,stop_items,0.692175,False
19,u00068,이제 근처 화장실 어디예요,other,test,template,clear,stop_items,0.606073,False
22,u00095,젤리 여기 들어가는 거 맞아,stop_items,test,template,hard,report_issue,0.862018,False
35,u00199,이제 근처 화장실 어디야,other,test,template,clear,stop_items,0.444299,False
45,u00252,그 물건 확인할게 빨리,scan,test,template,clear,stop_items,0.646834,False
48,u00278,저기 다 넣어놨어요,complete,test,template,hard,scan,0.755674,False
49,u00290,근처 화장실 어디야 얼른,other,test,template,clear,stop_items,0.426532,False
51,u00295,제육 도시락 여기 들어가는 거 맞아,stop_items,test,template,hard,report_issue,0.715838,False
55,u00322,자 근처 화장실 어디야,other,test,template,clear,stop_items,0.520354,False


In [19]:
def slot_pairs(call):
    return {(k, str(v)) for k, v in call.items() if k not in ("fn", "_bad")}


def score_fc(df):
    """df: text, target, pred_raw 열 -> 지표 dict와 행별 결과"""
    rows = []
    for gold_s, raw in zip(df.target, df.pred_raw):
        gold, pred = json.loads(gold_s), parse_call(raw)
        g_res, p_res = resolve(gold, CURRENT_STORE), resolve(pred, CURRENT_STORE)
        rows.append({
            "valid_json": not pred.get("_bad", False),
            "fn_ok": pred.get("fn") == gold["fn"],
            "exact": {k: v for k, v in pred.items() if k != "_bad"} == gold,
            "tp": len(slot_pairs(pred) & slot_pairs(gold)),
            "n_pred": len(slot_pairs(pred)), "n_gold": len(slot_pairs(gold)),
            "final_ok": g_res == p_res,            # 보정 후 실행 내용(또는 되묻기 여부)까지 같은가
        })
    r = pd.DataFrame(rows, index=df.index)
    tp, npred, ngold = r.tp.sum(), r.n_pred.sum(), r.n_gold.sum()
    prec, rec = tp / max(npred, 1), tp / max(ngold, 1)
    summary = {"n": len(r), "json_valid": r.valid_json.mean(), "fn_acc": r.fn_ok.mean(), "exact": r.exact.mean(),
               "slot_f1": 2 * prec * rec / max(prec + rec, 1e-9), "final_ok": r.final_ok.mean()}
    return summary, r

In [20]:
# ▶ GPU 서버에서 실행
te_f = fc_df[fc_df.split == "test"].reset_index(drop=True)
t0 = time.time()
te_f["pred_raw"] = fc_generate(te_f.text.tolist())
sec = time.time() - t0
summary, r = score_fc(te_f)
te_f = pd.concat([te_f, r], axis=1)
print(f"[function calling] 생성 시간 {sec:.0f}초 ({sec / len(te_f) * 1000:.0f} ms/문장, 서버 GPU 배치 기준)")
print({k: round(v, 3) if isinstance(v, float) else v for k, v in summary.items()})
print(te_f.groupby(["source", "ambiguity"])[["fn_ok", "exact", "final_ok"]].mean().round(3))
print(te_f.groupby(te_f.target.map(lambda s: json.loads(s)["fn"]))[["fn_ok", "exact", "final_ok"]].mean().round(3))

# 두 모듈의 fn 판단 비교
both = te_f.assign(intent_pred=te_i.set_index("id").loc[te_f.id, "pred"].values)
print("의도분류와 LLM의 fn이 같은 비율:", (both.intent_pred == both.pred_raw.map(lambda s: parse_call(s)["fn"])).mean().round(3))
te_f[~te_f.final_ok][["text", "target", "pred_raw", "ambiguity", "source"]].head(20)

[function calling] 생성 시간 115초 (266 ms/문장, 서버 GPU 배치 기준)
{'n': 431, 'json_valid': np.float64(1.0), 'fn_acc': np.float64(0.91), 'exact': np.float64(0.863), 'slot_f1': np.float64(0.935), 'final_ok': np.float64(0.868)}
                    fn_ok  exact  final_ok
source   ambiguity                        
gemini   ask        1.000  1.000     1.000
         clear      0.970  0.896     0.896
template ask        1.000  0.833     1.000
         clear      0.905  0.861     0.861
         hard       0.833  0.833     0.833
              fn_ok  exact  final_ok
target                              
complete      1.000  0.986     0.986
next_stop     0.770  0.770     0.770
other         0.986  0.986     0.986
report_issue  0.870  0.623     0.652
scan          1.000  1.000     1.000
stop_items    0.840  0.813     0.813
의도분류와 LLM의 fn이 같은 비율: 0.807


,text,target,pred_raw,ambiguity,source
1,저기 참치삼각김밥 박스 하나 수량이 안 맞아,"{""fn"":""report_issue"",""item"":""참치삼각김밥"",""issue"":""...","{""fn"":""report_issue"",""item"":""참치삼각김밥"",""issue"":""...",clear,template
2,바나나우유 박스 하나 새고 있어 빨리,"{""fn"":""report_issue"",""item"":""바나나우유"",""issue"":""파손""}","{""fn"":""report_issue"",""item"":""바나나우유"",""issue"":""파...",clear,template
14,서초점 라면 한 박스 모자라요,"{""fn"":""report_issue"",""store"":""서초점"",""item"":""라면""...","{""fn"":""report_issue"",""store"":""서초점"",""item"":""라면""...",clear,gemini
18,자 몇 개 남았지,"{""fn"":""next_stop""}","{""fn"":""stop_items""}",clear,template
26,바나나우유 상태 좀 봐야겠다 좀,"{""fn"":""report_issue"",""item"":""바나나우유""}","{""fn"":""stop_items"",""item"":""바나나우유""}",hard,template
27,몇 개 남았지,"{""fn"":""next_stop""}","{""fn"":""stop_items""}",clear,template
33,신림점 치킨마요 하나 모자람,"{""fn"":""report_issue"",""store"":""신림점"",""item"":""치킨마...","{""fn"":""report_issue"",""store"":""신림점"",""item"":""치킨마...",clear,gemini
84,바나나우유 박스 하나 엉뚱한 게 실렸어,"{""fn"":""report_issue"",""item"":""바나나우유"",""issue"":""오...","{""fn"":""report_issue"",""item"":""바나나우유"",""issue"":""오...",clear,template
88,아 박스 진짜 무게 나가네,"{""fn"":""other""}","{""fn"":""report_issue"",""issue"":""수량부족""}",clear,gemini
90,요구르트 박스 하나 온도가 높아,"{""fn"":""report_issue"",""item"":""요구르트"",""issue"":""온도...","{""fn"":""report_issue"",""item"":""요구르트"",""issue"":""온도...",clear,template


In [21]:
# ▶ GPU 서버에서 실행
# 결과 저장 (주간보고 '데이터셋 구축 및 성능 개선 결과', '문제점 분석'에 그대로 사용)
te_i.to_csv(f"{OUT_DIR}/test_pred_intent.csv", index=False, encoding="utf-8-sig")
te_f.to_csv(f"{OUT_DIR}/test_pred_fc.csv", index=False, encoding="utf-8-sig")
with open(f"{OUT_DIR}/test_summary.json", "w", encoding="utf-8") as f:
    json.dump({"intent": {"acc": float(te_i.correct.mean()),
                          "macro_f1": float(f1_score(te_i.label, te_i.pred, average="macro"))},
               "function_calling": {k: float(v) for k, v in summary.items()}}, f, ensure_ascii=False, indent=2)
print(open(f"{OUT_DIR}/test_summary.json", encoding="utf-8").read())

{
  "intent": {
    "acc": 0.8932714617169374,
    "macro_f1": 0.8942139573545728
  },
  "function_calling": {
    "n": 431.0,
    "json_valid": 1.0,
    "fn_acc": 0.9095127610208816,
    "exact": 0.8631090487238979,
    "slot_f1": 0.934931506849315,
    "final_ok": 0.8677494199535963
  }
}


## 5. 완성 파이프라인: 직접 입력해서 테스트

```
문장 ─▶ ① 의도분류 ─┬─ next_stop / scan / other (확신 0.9 이상) ─────────────┐
                    └─ 그 외 ─▶ ② LLM function calling ─▶ JSON              │
                                         ▼                                  ▼
                              ③ 값 보정 (DB 맞추기·되묻기) ─▶ ④ 실행 (가상 DB) ─▶ ⑤ 응답 문장 (템플릿)
```

- ①은 빠른 길: 값이 필요 없는 명령은 LLM을 부르지 않습니다.
- ③, ④, ⑤는 모델이 아니라 규칙 코드라서 학습 없이 바로 동작합니다. ⑤의 문장은 그대로 TTS에 넘기면 됩니다.
- 되묻기 뒤 사용자가 "서울우유"처럼 답하면, 앞의 JSON에 그 값을 채워 다시 보정합니다 (아래 `Assistant` 클래스).
- 가상 DB는 앱/백엔드 팀이 실제 API로 바꿀 자리입니다.

In [22]:
# 가상 DB (오늘 배송 순서와 점포별 납품 품목)
ROUTE = ["역삼점", "서초점", "방배점", "반포점", "도곡점"]
STOP_ITEMS = {
    "역삼점": {"서울우유": (12, "개"), "바나나우유": (6, "개"), "제육도시락": (4, "개"), "생수": (2, "박스")},
    "서초점": {"딸기우유": (6, "개"), "참치삼각김밥": (10, "개"), "콜라": (1, "박스")},
    "방배점": {"샌드위치": (8, "개"), "캔커피": (2, "박스"), "초코파이": (3, "박스")},
    "반포점": {"치킨마요도시락": (5, "개"), "이온음료": (1, "박스"), "아이스크림": (2, "박스")},
    "도곡점": {"컵라면": (2, "박스"), "새우깡": (10, "봉지"), "요거트": (12, "개")},
}
ISSUE_LOG = []


def josa(word, pair):
    """받침 여부에 따라 조사 선택: josa('역삼점', '은/는') -> '역삼점은'"""
    a, b = pair.split("/")
    c = ord(word[-1]) - 0xAC00
    return word + (a if 0 <= c < 11172 and c % 28 else b)


class Assistant:
    def __init__(self, intent_fn=None, fc_fn=None, current_store="역삼점", skip_conf=0.9):
        self.intent_fn, self.fc_fn = intent_fn, fc_fn
        self.state = {"current": current_store, "done": []}
        self.skip_conf = skip_conf
        self.pending = None          # 되묻는 중인 JSON

    # ④ 실행 + ⑤ 응답 문장
    def execute(self, r):
        fn, st = r["fn"], self.state
        if fn == "next_stop":
            left = [s for s in ROUTE if s not in st["done"]]
            if not left:
                return "오늘 배송은 모두 끝났어요. 수고하셨어요."
            nxt = [s for s in left if s != st["current"]] or left
            return f"다음은 {josa(nxt[0], '이에요/예요')}. 남은 곳은 {len(left)}곳이에요."
        if fn == "stop_items":
            items = STOP_ITEMS.get(r["store"], {})
            if r.get("item"):
                if r["item"] not in items:
                    return f"{r['store']}에는 {r['item']} 납품이 없어요."
                q, u = items[r["item"]]
                return f"{r['store']} {josa(r['item'], '은/는')} {josa(f'{q}{u}', '이에요/예요')}."
            if not items:
                return f"{r['store']} 납품 목록이 없어요."
            return f"{r['store']} 납품은 " + josa(", ".join(f"{k} {q}{u}" for k, (q, u) in items.items()), "이에요/예요") + "."
        if fn == "complete":
            if r["store"] not in st["done"]:
                st["done"].append(r["store"])
            who = f" 인수자는 {josa(r['receiver'], '이에요/예요')}." if r.get("receiver") else ""
            return f"{r['store']} 납품 완료로 기록했어요.{who}"
        if fn == "report_issue":
            ISSUE_LOG.append(dict(r))
            qty = f" {r['qty']}{r['unit']}" if r.get("qty") else ""
            return f"{r['store']} {r['item']}{qty} {josa(r['issue'], '으로/로')} 기록했어요."
        if fn == "scan":
            return "스캔 화면을 열게요."           # 앱이 카메라 화면을 띄우는 신호
        return "죄송해요, 배송 업무만 도와드릴 수 있어요."

    def __call__(self, text):
        t = {"text": text}
        t0 = time.time()
        # 되묻기에 대한 답이면 앞의 JSON을 채워서 다시 보정
        if self.pending:
            call = dict(self.pending)
            miss = self.pending_slot
            self.pending = None
            if miss == "issue":
                ans = next((k for k, ps in ISSUE_PHRASES.items() if k in text or any(p in text for p in ps)), None)
                call["issue"] = ans
            else:
                call[miss] = text.strip()
            t["route"] = "되묻기 답"
        else:
            fn, conf = self.intent_fn(text) if self.intent_fn else ("?", 0.0)
            t["intent"] = f"{fn} ({conf:.2f})"
            t["intent_ms"] = round((time.time() - t0) * 1000)
            if fn in ("next_stop", "scan", "other") and conf >= self.skip_conf:
                call, t["route"] = {"fn": fn}, "의도분류만 (LLM 생략)"
            else:
                t1 = time.time()
                raw = self.fc_fn(text)
                call = parse_call(raw)
                t["llm_raw"], t["llm_ms"] = raw, round((time.time() - t1) * 1000)
                t["route"] = "LLM function calling"
        t["call"] = {k: v for k, v in call.items() if k != "_bad"}
        r = resolve(call, self.state["current"])
        if not r["ok"]:
            self.pending, self.pending_slot = call, r["slot"]   # 다음 말을 이 값의 답으로 받음
            t["reply"] = r["ask"]
        else:
            if r["fn"] in ("stop_items", "complete") and r.get("store"):
                self.state["current"] = r["store"]
            t["reply"] = self.execute(r)
        t["total_ms"] = round((time.time() - t0) * 1000)
        return t


def show(t):
    print(f"🗣  {t['text']}")
    for k in ("route", "intent", "llm_raw", "call"):
        if k in t:
            print(f"   {k:8}: {t[k]}")
    print(f"🤖 {t['reply']}   ({t['total_ms']} ms)\n")

In [23]:
# 학습한 모델 연결 (2~3단계를 실행했으면 그대로 동작)
if "intent_predict" in globals() and "fc_generate" in globals():
    bot = Assistant(intent_fn=lambda s: intent_predict([s])[0], fc_fn=lambda s: fc_generate([s])[0])
else:
    print("학습된 모델이 없어 규칙만 확인하는 가짜 모델로 연결합니다 (JSON을 직접 입력하면 그 JSON을 LLM 출력으로 사용).")
    bot = Assistant(intent_fn=lambda s: ("report_issue", 0.5), fc_fn=lambda s: s)

DEMO = ["다음 어디야", "여기 뭐 내려야 돼", "우유 두 개 터졌어", "서울우유", "점장님한테 넘겼어",
        "서초점 콜라 몇 박스야", "이거 미지근해", "바나나우유", "다음 어디 가야 하지", "노래 틀어줘"]
if "fc_generate" not in globals():   # 가짜 모델일 때는 LLM이 냈을 JSON을 대신 넣어 흐름만 확인
    DEMO = ['{"fn":"next_stop"}', '{"fn":"stop_items"}',
            '{"fn":"report_issue","item":"우유","issue":"파손","qty":2,"unit":"개"}', "서울우유",
            '{"fn":"complete","receiver":"점장님"}', '{"fn":"stop_items","store":"서초점","item":"콜라"}',
            '{"fn":"report_issue","issue":"온도이상"}', "바나나우유", '{"fn":"next_stop"}', '{"fn":"other"}']
for s in DEMO:
    show(bot(s))
print("기록된 이상 보고:", ISSUE_LOG)

🗣  다음 어디야
   route   : 의도분류만 (LLM 생략)
   intent  : next_stop (0.97)
   call    : {'fn': 'next_stop'}
🤖 다음은 서초점이에요. 남은 곳은 5곳이에요.   (118 ms)



🗣  여기 뭐 내려야 돼
   route   : LLM function calling
   intent  : stop_items (0.95)
   llm_raw : {"fn":"stop_items"}
   call    : {'fn': 'stop_items'}
🤖 역삼점 납품은 서울우유 12개, 바나나우유 6개, 제육도시락 4개, 생수 2박스예요.   (691 ms)



🗣  우유 두 개 터졌어
   route   : LLM function calling
   intent  : report_issue (0.94)
   llm_raw : {"fn":"report_issue","item":"우유","issue":"파손","qty":2,"unit":"개"}
   call    : {'fn': 'report_issue', 'item': '우유', 'issue': '파손', 'qty': 2, 'unit': '개'}
🤖 서울우유, 바나나우유, 딸기우유 중 어떤 건가요?   (2769 ms)

🗣  서울우유
   route   : 되묻기 답
   call    : {'fn': 'report_issue', 'item': '서울우유', 'issue': '파손', 'qty': 2, 'unit': '개'}
🤖 역삼점 서울우유 2개 파손으로 기록했어요.   (1 ms)



🗣  점장님한테 넘겼어
   route   : LLM function calling
   intent  : complete (0.97)
   llm_raw : {"fn":"complete","receiver":"점장님"}
   call    : {'fn': 'complete', 'receiver': '점장님'}
🤖 역삼점 납품 완료로 기록했어요. 인수자는 점장님이에요.   (1161 ms)



🗣  서초점 콜라 몇 박스야
   route   : LLM function calling
   intent  : stop_items (0.94)
   llm_raw : {"fn":"stop_items","store":"서초점","item":"콜라"}
   call    : {'fn': 'stop_items', 'store': '서초점', 'item': '콜라'}
🤖 서초점 콜라는 1박스예요.   (1540 ms)



🗣  이거 미지근해
   route   : LLM function calling
   intent  : report_issue (0.90)
   llm_raw : {"fn":"report_issue","issue":"온도이상"}
   call    : {'fn': 'report_issue', 'issue': '온도이상'}
🤖 어떤 품목인가요?   (1072 ms)

🗣  바나나우유
   route   : 되묻기 답
   call    : {'fn': 'report_issue', 'issue': '온도이상', 'item': '바나나우유'}
🤖 서초점 바나나우유 온도이상으로 기록했어요.   (0 ms)

🗣  다음 어디 가야 하지
   route   : 의도분류만 (LLM 생략)
   intent  : next_stop (0.97)
   call    : {'fn': 'next_stop'}
🤖 다음은 방배점이에요. 남은 곳은 4곳이에요.   (11 ms)

🗣  노래 틀어줘
   route   : 의도분류만 (LLM 생략)
   intent  : other (0.95)
   call    : {'fn': 'other'}
🤖 죄송해요, 배송 업무만 도와드릴 수 있어요.   (12 ms)

기록된 이상 보고: [{'ok': True, 'fn': 'report_issue', 'store': '역삼점', 'item': '서울우유', 'issue': '파손', 'qty': 2, 'unit': '개'}, {'ok': True, 'fn': 'report_issue', 'store': '서초점', 'item': '바나나우유', 'issue': '온도이상'}]


In [24]:
# 직접 입력해 보기: RUN_INTERACTIVE=True로 바꾸고 직접 실행하면 콘솔에서 입력할 수 있습니다.
# (노트북을 한 번에 실행(nbconvert 등)할 때는 input()이 멈추므로 기본값 False로 건너뜁니다.)
RUN_INTERACTIVE = False
if RUN_INTERACTIVE:
    while True:
        s = input("발화> ").strip()
        if not s:
            break
        show(bot(s))
else:
    print("RUN_INTERACTIVE=False: 대화형 입력 테스트를 건너뜁니다. (직접 실행 시 True로 바꾸세요)")


RUN_INTERACTIVE=False: 대화형 입력 테스트를 건너뜁니다. (직접 실행 시 True로 바꾸세요)


## 6. (선택) 음성 인식 붙이기: SenseVoice + sherpa-onnx

**추천: SenseVoice-Small int8 (sherpa-onnx)**. 강의 자료(Part 3)가 이 조합을 쓰고, 보드 배포 이미지의 `~/voice/models`에 이미 들어 있습니다. 비자동회귀 방식이라 문장 전체를 한 번에 인식해서 Whisper보다 훨씬 빠르고, 한국어를 지원하며, `use_itn=True`로 "두 개"를 "2개"처럼 숫자로 바꿔 줍니다. 보드에서는 CPU 4스레드로 돌리고 NPU는 LLM 전용으로 둡니다.

| 모델 | 장점 | 단점 |
| --- | --- | --- |
| **SenseVoice-Small int8** (추천) | 빠름, 한국어 지원, 강의·보드에 이미 준비됨 | 한국어 전용 모델보다 고유명사 오인식 가능 |
| Whisper base/small | 한국어 품질 무난, 자료 많음 | 자동회귀라 보드 CPU에서 느림 |
| sherpa-onnx 한국어 zipformer (스트리밍) | 말하는 중에 바로 글자가 나옴 | 별도로 받아야 하고 설정이 더 필요 |

점포·품목 이름 오인식(예: "약삼졈")은 3단계 데이터의 STT 노이즈와 ③ 값 보정의 발음 유사도 비교가 받아 줍니다.

아래 셀은 PC나 서버에서 wav 파일 하나로 시험하는 코드입니다 (16kHz 모노 wav 권장).

In [25]:
# ▶ 모델 다운로드 후 실행 (GPU 필요 없음)
# !pip install -q sherpa-onnx soundfile
# !wget -q https://github.com/k2-fsa/sherpa-onnx/releases/download/asr-models/sherpa-onnx-sense-voice-zh-en-ja-ko-yue-2024-07-17.tar.bz2
# !tar xf sherpa-onnx-sense-voice-zh-en-ja-ko-yue-2024-07-17.tar.bz2
SV_DIR = "sherpa-onnx-sense-voice-zh-en-ja-ko-yue-2024-07-17"
WAV = "test.wav"     # 휴대폰으로 녹음한 파일을 16kHz wav로 바꿔 올리세요

if not os.path.exists(SV_DIR):
    print(f"{SV_DIR} 없음 -> SenseVoice 모델을 받지 않아 이 셀은 건너뜁니다. (주석의 wget/tar 참고)")
elif not os.path.exists(WAV):
    print(f"{WAV} 없음 -> STT 테스트를 건너뜁니다.")
else:
    import sherpa_onnx, soundfile as sf

    asr = sherpa_onnx.OfflineRecognizer.from_sense_voice(
        model=f"{SV_DIR}/model.int8.onnx", tokens=f"{SV_DIR}/tokens.txt",
        language="ko", use_itn=True, num_threads=4)

    def stt(wav_path):
        audio, sr = sf.read(wav_path, dtype="float32")
        if audio.ndim > 1:
            audio = audio.mean(axis=1)
        s = asr.create_stream()
        s.accept_waveform(sr, audio)
        asr.decode_stream(s)
        return s.result.text.strip()

    t0 = time.time()
    text = stt(WAV)
    print(f"STT: {text}  ({(time.time() - t0) * 1000:.0f} ms)")
    show(bot(text))


sherpa-onnx-sense-voice-zh-en-ja-ko-yue-2024-07-17 없음 -> SenseVoice 모델을 받지 않아 이 셀은 건너뜁니다. (주석의 wget/tar 참고)
